# ShopMind — Build Indexes (BU SCC)
**Run this notebook on BU SCC (GPU node preferred, CPU works too).**

What this notebook builds:
1. **Chroma DB** — BGE-large embeddings of 100K reviews (for RAG)
2. **FAISS index** — product text embeddings for semantic search

At the end, zip `outputs/` and download it to your Mac.

**Estimated time:** 1-2 hours on GPU, 3-4 hours on CPU

## 0. Install Dependencies

In [1]:
import subprocess, sys

def pip(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + pkg.split())

pip('torch --index-url https://download.pytorch.org/whl/cu118')
pip('sentence-transformers==2.7.0')
pip('chromadb==0.5.0')
pip('faiss-gpu==1.7.2')
pip('datasets==2.18.0')
pip('pandas numpy tqdm')
pip('pysqlite3-binary')
print('✓ All packages installed')


[notice] A new release of pip is available: 23.2.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 23.2.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 23.2.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 23.2.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 23.2.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


✓ All packages installed



[notice] A new release of pip is available: 23.2.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


## 1. Setup

In [2]:
import os, json
import numpy as np
import pandas as pd
from tqdm import tqdm
import torch

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

os.makedirs('outputs/chroma_db', exist_ok=True)
os.makedirs('outputs/faiss_index', exist_ok=True)
os.makedirs('data', exist_ok=True)
print('✓ Directories ready')

Device: cuda
GPU: NVIDIA H200 NVL
✓ Directories ready


## 2. Download & Parse Data
Skip this section if you already ran `scc_train_models.ipynb` and have `data/reviews.csv`.

In [3]:
from datasets import load_dataset

MAX_REVIEWS  = 100_000
MAX_PRODUCTS = 50_000

print('Streaming reviews from HuggingFace...')
rev_ds = load_dataset(
    'McAuley-Lab/Amazon-Reviews-2023',
    'raw_review_Electronics',
    split='full',
    streaming=True,
    trust_remote_code=True
)
rev_rows = []
for i, row in enumerate(rev_ds):
    if i >= MAX_REVIEWS: break
    rev_rows.append(row)
    if (i+1) % 10000 == 0: print(f'  {i+1:,} reviews...')

print('Streaming product metadata from HuggingFace...')
meta_ds = load_dataset(
    'McAuley-Lab/Amazon-Reviews-2023',
    'raw_meta_Electronics',
    split='full',
    streaming=True,
    trust_remote_code=True
)
meta_rows = []
for i, row in enumerate(meta_ds):
    if i >= MAX_PRODUCTS: break
    meta_rows.append(row)
    if (i+1) % 10000 == 0: print(f'  {i+1:,} products...')

print(f'✓ Got {len(rev_rows):,} reviews and {len(meta_rows):,} products')

Streaming reviews from HuggingFace...
  10,000 reviews...
  20,000 reviews...
  30,000 reviews...
  40,000 reviews...
  50,000 reviews...
  60,000 reviews...
  70,000 reviews...
  80,000 reviews...
  90,000 reviews...
  100,000 reviews...
Streaming product metadata from HuggingFace...
  10,000 products...
  20,000 products...
  30,000 products...
  40,000 products...
  50,000 products...
✓ Got 100,000 reviews and 50,000 products


In [4]:
# Build reviews DataFrame
df_rev = pd.DataFrame(rev_rows)
rename_rev = {'parent_asin':'product_id','text':'reviewText','rating':'overall','helpful_vote':'helpful_votes'}
df_rev = df_rev.rename(columns={k:v for k,v in rename_rev.items() if k in df_rev.columns})
keep = ['product_id','user_id','overall','reviewText','verified_purchase','helpful_votes']
df_rev = df_rev[[c for c in keep if c in df_rev.columns]].dropna(subset=['product_id','reviewText'])
df_rev['reviewText'] = df_rev['reviewText'].astype(str)
df_rev = df_rev[df_rev['reviewText'].str.len() >= 20].reset_index(drop=True)
df_rev.to_csv('data/reviews.csv', index=False)
print(f'✓ {len(df_rev):,} reviews saved')

# Build products DataFrame
df_prod = pd.DataFrame(meta_rows)
rename_prod = {'parent_asin':'product_id','main_category':'category','store':'brand','average_rating':'avg_rating','rating_number':'review_count'}
df_prod = df_prod.rename(columns={k:v for k,v in rename_prod.items() if k in df_prod.columns})
if 'description' in df_prod.columns:
    df_prod['description'] = df_prod['description'].apply(lambda x: ' '.join(x) if isinstance(x, list) else str(x or ''))
if 'price' in df_prod.columns:
    df_prod['price'] = pd.to_numeric(df_prod['price'], errors='coerce').fillna(0)
valid_ids = set(df_rev['product_id'].unique())
df_prod = df_prod[df_prod['product_id'].isin(valid_ids)]
df_prod.to_csv('data/products.csv', index=False)
print(f'✓ {len(df_prod):,} products saved')

✓ 92,487 reviews saved
✓ 3,607 products saved


## 3. Load Embedding Model

In [5]:
from sentence_transformers import SentenceTransformer

# BGE-large: 1024-dim, best for retrieval
print('Loading BGE-large embedding model...')
embed_model = SentenceTransformer('BAAI/bge-large-en-v1.5', device=DEVICE)
print(f'✓ Model loaded on {DEVICE}')

def embed_batch(texts, batch_size=128, prefix=''):
    """Embed a list of texts in batches."""
    if prefix:
        texts = [prefix + t for t in texts]
    return embed_model.encode(
        texts,
        batch_size=batch_size,
        show_progress_bar=True,
        normalize_embeddings=True,
        device=DEVICE
    )

/restricted/projectnb/dentalds/qwen_env_pkgs/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


Loading BGE-large embedding model...
✓ Model loaded on cuda


## 4. Build Chroma Review Store (for RAG)

In [6]:
# Fix for old sqlite3 on SCC
__import__('pysqlite3')
import sys
sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

import chromadb

CHROMA_PATH = 'outputs/chroma_db'
BATCH_SIZE = 512

client = chromadb.PersistentClient(path=CHROMA_PATH)

# Delete existing collection if re-running
try:
    client.delete_collection('reviews')
    print('Deleted existing reviews collection')
except:
    pass

collection = client.create_collection(
    name='reviews',
    metadata={'hnsw:space': 'cosine'}
)

# Filter valid reviews
valid = df_rev[df_rev['reviewText'].str.len() >= 20].copy()
valid = valid.head(100_000)
print(f'Indexing {len(valid):,} reviews into Chroma...')

texts, metas, ids = [], [], []
for i, row in tqdm(valid.iterrows(), total=len(valid)):
    text = str(row['reviewText'])[:1000]
    texts.append(text)
    metas.append({
        'product_id': str(row.get('product_id', '')),
        'rating': float(row.get('overall', 3.0)),
        'verified': int(row.get('verified_purchase', 1)),
    })
    ids.append(f'rev_{i}')

    if len(texts) >= BATCH_SIZE:
        embeddings = embed_batch(texts, prefix='Represent this review: ')
        collection.add(documents=texts, embeddings=embeddings.tolist(), metadatas=metas, ids=ids)
        texts, metas, ids = [], [], []

if texts:
    embeddings = embed_batch(texts, prefix='Represent this review: ')
    collection.add(documents=texts, embeddings=embeddings.tolist(), metadatas=metas, ids=ids)

print(f'✓ Chroma DB built: {collection.count():,} documents')

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Indexing 92,487 reviews into Chroma...


  0%|          | 0/92487 [00:00<?, ?it/s]

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Failed to send telemetry event CollectionAddEvent: capture() takes 1 positional argument but 3 were given
  1%|          | 512/92487 [00:02<06:43, 228.20it/s]

...

 99%|█████████▉| 91648/92487 [05:10<00:03, 234.65it/s]

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

100%|██████████| 92487/92487 [05:11<00:00, 296.58it/s]


Batches:   0%|          | 0/3 [00:00<?, ?it/s]

✓ Chroma DB built: 92,487 documents


## 5. Build FAISS Product Index (for Semantic Search)

In [7]:
import faiss

FAISS_PATH = 'outputs/faiss_index'

# Build text for each product: title + description + category
df_prod_clean = df_prod.dropna(subset=['product_id']).copy()
df_prod_clean['search_text'] = (
    df_prod_clean.get('title', pd.Series([''] * len(df_prod_clean))).fillna('') + ' ' +
    df_prod_clean.get('description', pd.Series([''] * len(df_prod_clean))).fillna('').str[:200] + ' ' +
    df_prod_clean.get('category', pd.Series([''] * len(df_prod_clean))).fillna('')
).str.strip()

product_texts = df_prod_clean['search_text'].tolist()
product_ids   = df_prod_clean['product_id'].tolist()

print(f'Embedding {len(product_texts):,} products...')
product_embeddings = embed_batch(product_texts, batch_size=256, prefix='Represent this product: ')
print(f'✓ Embeddings shape: {product_embeddings.shape}')

# Build IVF FAISS index
dim = product_embeddings.shape[1]  # 1024
n_lists = min(int(np.sqrt(len(product_texts))), 256)

quantizer = faiss.IndexFlatIP(dim)
index = faiss.IndexIVFFlat(quantizer, dim, n_lists, faiss.METRIC_INNER_PRODUCT)
index.train(product_embeddings.astype('float32'))
index.add(product_embeddings.astype('float32'))
index.nprobe = 10

# Save index + product ID mapping
faiss.write_index(index, f'{FAISS_PATH}/product.index')
with open(f'{FAISS_PATH}/product_ids.json', 'w') as f:
    json.dump(product_ids, f)

print(f'✓ FAISS index saved: {index.ntotal:,} vectors at {FAISS_PATH}/')

Embedding 3,607 products...


Batches:   0%|          | 0/15 [00:00<?, ?it/s]

✓ Embeddings shape: (3607, 1024)
✓ FAISS index saved: 3,607 vectors at outputs/faiss_index/


## 6. Quick Sanity Check

In [8]:
# Test Chroma
q_embed = embed_batch(['good battery life'], prefix='Query: ')
results = collection.query(query_embeddings=q_embed.tolist(), n_results=3)
print('=== Chroma test query: "good battery life" ===')
for doc in results['documents'][0]:
    print(f'  → {doc[:100]}...')

# Test FAISS
q_prod = embed_batch(['wireless noise cancelling headphones'], prefix='Query: ')
scores, indices = index.search(q_prod.astype('float32'), k=3)
print('\n=== FAISS test query: "wireless noise cancelling headphones" ===')
for idx, score in zip(indices[0], scores[0]):
    if idx >= 0:
        print(f'  [{score:.3f}] {product_texts[idx][:80]}...')

print('\n✓ Both indexes working!')

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


=== Chroma test query: "good battery life" ===
  → very good. everything is fine except the battery life....
  → Good sound<br />Long battery life...
  → Battery life is not great....


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


=== FAISS test query: "wireless noise cancelling headphones" ===
  [0.729] Otium Noise Cancelling Wireless in Ear Bluetooth with Mic Deep Bass, Foldable Co...
  [0.724] JVC Wireless Noise Canceling Over Ear Headphones, Bluetooth, Instant paring with...
  [0.722] Bose QuietComfort Noise Cancelling Earbuds-Bluetooth Wireless Earphones, Triple ...

✓ Both indexes working!


## 7. Package for Download

In [9]:
import shutil

# Save CSVs too (needed locally)
df_rev.to_csv('outputs/reviews.csv', index=False)
df_prod.to_csv('outputs/products.csv', index=False)

# Zip everything
shutil.make_archive('shopmind_indexes', 'zip', '.', 'outputs')
size_mb = os.path.getsize('shopmind_indexes.zip') / 1e6
print(f'✓ Created shopmind_indexes.zip ({size_mb:.0f} MB)')
print('Download from JupyterHub file browser → right-click → Download')
print()
print('Contents:')
for root, dirs, files in os.walk('outputs'):
    for f in files:
        fp = os.path.join(root, f)
        print(f'  {fp}  ({os.path.getsize(fp)/1e6:.1f} MB)')

✓ Created shopmind_indexes.zip (2999 MB)
Download from JupyterHub file browser → right-click → Download

Contents:
  outputs/reviews.csv  (41.9 MB)
  outputs/products.csv  (16.7 MB)
  outputs/chroma_db/chroma.sqlite3  (632.5 MB)
  outputs/chroma_db/67caf093-5d5a-40f6-9a0a-c183b56302df/header.bin  (0.0 MB)
  outputs/chroma_db/67caf093-5d5a-40f6-9a0a-c183b56302df/index_metadata.pickle  (3.0 MB)
  outputs/chroma_db/67caf093-5d5a-40f6-9a0a-c183b56302df/data_level0.bin  (389.7 MB)
  outputs/chroma_db/67caf093-5d5a-40f6-9a0a-c183b56302df/link_lists.bin  (0.8 MB)
  outputs/chroma_db/67caf093-5d5a-40f6-9a0a-c183b56302df/length.bin  (0.4 MB)
  outputs/models/fake_review_model.joblib  (0.3 MB)
  outputs/models/training_meta.json  (0.0 MB)
  outputs/models/absa_model/model.safetensors  (267.8 MB)
  outputs/models/absa_model/config.json  (0.0 MB)
  outputs/models/absa_model/training_args.bin  (0.0 MB)
  outputs/models/absa_model/tokenizer_config.json  (0.0 MB)
  outputs/models/absa_model/vocab.txt